In [ ]:
!wget https://files.grouplens.org/datasets/movielens/ml-32m.zip
!unzip ml-32m.zip

--2026-03-12 03:09:58--  https://files.grouplens.org/datasets/movielens/ml-32m.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 238950008 (228M) [application/zip]
Saving to: ‘ml-32m.zip’

ml-32m.zip          100%[===================>] 227.88M  29.4MB/s    in 8.5s    

2026-03-12 03:10:07 (26.7 MB/s) - ‘ml-32m.zip’ saved [238950008/238950008]

Archive:  ml-32m.zip
   creating: ml-32m/
  inflating: ml-32m/tags.csv         
  inflating: ml-32m/links.csv        
  inflating: ml-32m/README.txt       
  inflating: ml-32m/checksums.txt    
  inflating: ml-32m/ratings.csv      
  inflating: ml-32m/movies.csv       


In [ ]:
!head ml-32m/movies.csv

movieId,title,genres
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy
6,Heat (1995),Action|Crime|Thriller
7,Sabrina (1995),Comedy|Romance
8,Tom and Huck (1995),Adventure|Children
9,Sudden Death (1995),Action


In [ ]:
!head ml-32m/tags.csv

userId,movieId,tag,timestamp
22,26479,Kevin Kline,1583038886
22,79592,misogyny,1581476297
22,247150,acrophobia,1622483469
34,2174,music,1249808064
34,2174,weird,1249808102
34,8623,Steve Martin,1249808497
55,5766,the killls and the score,1319322078
58,7451,bullying,1672551536
58,7451,clique,1672551510


# Tag가 유사한 영화 찾기

영화 제목과 TAG 불러오기

In [ ]:
import csv

movies = {}

with open('ml-32m/movies.csv','r') as f:
  reader = csv.reader(f)
  next(reader) # 첫 번째 줄 패스

  for mid, title, _ in reader:
    movies[int(mid)] = {'title':title}



In [ ]:
for movie in movies.values():
  movie['tags'] = set()

with open('ml-32m/tags.csv', 'r') as f:
  reader = csv.reader(f)
  next(reader)

  for _, mid, tag, _ in reader:
    movies[int(mid)]['tags'].add(tag.lower().strip())

In [ ]:
movie

{'title': 'Race to the Summit (2023)', 'tags': set()}

자카드 유사도가 가장 높은 k개의 영화 찾기

In [ ]:
from tqdm import tqdm

def jaccard_similarity(a,b):
  union_size = len(a | b) # 합집합
  intersection_size = len(a & b) # 교집합

  if union_size == 0:
    return 0

  return intersection_size / union_size

def get_topk(target_mid, k=20, key='tags',sim_func = jaccard_similarity):
  target_set = movies[target_mid][key]

  res = []

  for mid, movie in tqdm(movies.items()):
    if mid == target_mid: continue
    compare_set = movie[key]
    score = sim_func(target_set, compare_set)
    res.append((score,movies[mid]['title']))
  res.sort(reverse = True)
  return res[:k]


In [ ]:
# 이 추천시스템은 어떤 영화가 입력으로 주어지면 tag 집합의 jaccard similarity가 높은 순서로 영화를 추천합니다.
get_topk(164909,10) # 라라랜드와 태그가 유사한 영화

[(0.085, '(500) Days of Summer (2009)'),
 (0.0728476821192053, 'Blue Valentine (2010)'),
 (0.06486486486486487, 'Moulin Rouge (2001)'),
 (0.06134969325153374, 'Misérables, Les (2012)'),
 (0.05952380952380952, 'Brokeback Mountain (2005)'),
 (0.059113300492610835, 'Almost Famous (2000)'),
 (0.058823529411764705, 'Notebook, The (2004)'),
 (0.058394160583941604, 'Across the Universe (2007)'),
 (0.05699481865284974, 'Silver Linings Playbook (2012)'),
 (0.056962025316455694, 'Once (2006)')]

# 다른 사용자가 함께 본 영화 찾기

In [ ]:
!head ml-32m/ratings.csv

userId,movieId,rating,timestamp
1,17,4.0,944249077
1,25,1.0,944250228
1,29,2.0,943230976
1,30,5.0,944249077
1,32,5.0,943228858
1,34,2.0,943228491
1,36,1.0,944249008
1,80,5.0,944248943
1,110,3.0,943231119


In [ ]:
for movie in movies.values():
  movie['users'] = set()

with open('ml-32m/ratings.csv', 'r') as f:
  reader = csv.reader(f)
  next(reader)
  for uid, mid, rating, _ in reader:
    movies[int(mid)]['users'].add(int(uid))

In [ ]:
movie

{'title': 'Race to the Summit (2023)',
 'tags': set(),
 'users': {171418},
 'ratings': {171418: 3.5}}

In [ ]:
# 이 추천시스템은 어떤 영화가 입력으로 주어지면 이 영화에 별점을 매긴 사용자 집합과 jaccard similarity가 높은 영화를 순서대로 추천합니다.
# Interstellar (2014) 와 유사한 영화 출력
get_topk(109487,k=20,key='users')

100%|██████████| 87585/87585 [09:06<00:00, 160.37it/s]


[(0.4911320547601499, 'Inception (2010)'),
 (0.4889573351585672, 'The Martian (2015)'),
 (0.432741878713845, 'Django Unchained (2012)'),
 (0.4208222867483062, 'Dark Knight Rises, The (2012)'),
 (0.40813108460167286, 'The Imitation Game (2014)'),
 (0.4065562244584084, 'Wolf of Wall Street, The (2013)'),
 (0.40598281156794735, 'Guardians of the Galaxy (2014)'),
 (0.40358711779594447, 'Dark Knight, The (2008)'),
 (0.3929898200414918, 'Ex Machina (2015)'),
 (0.3843366168065183, 'Edge of Tomorrow (2014)'),
 (0.3824597279025123, 'Shutter Island (2010)'),
 (0.38006561401024336, 'Mad Max: Fury Road (2015)'),
 (0.3763492215111281, 'Inglourious Basterds (2009)'),
 (0.3761336977289767, 'Arrival (2016)'),
 (0.3660013243780153, 'Avatar (2009)'),
 (0.3582690892186108, 'Prestige, The (2006)'),
 (0.3555301765791092, 'WALL·E (2008)'),
 (0.35447386952903265, 'Avengers, The (2012)'),
 (0.35138042309071355, 'Deadpool (2016)'),
 (0.3494290759270159, 'Gone Girl (2014)')]

# 비슷한 평가를 받는 영화 찾기

In [ ]:
for movie in movies.values():
  movie['ratings'] = {}

with open('ml-32m/ratings.csv', 'r') as f:
  reader = csv.reader(f)
  next(reader)
  for uid, mid, rating, _ in reader:
    movies[int(mid)]['ratings'][int(uid)] = float(rating)

for movie in tqdm(movies.values()):
  if len(movie['ratings']) == 0:
    continue
  avg = sum(movie['ratings'].values()) / len(movie['ratings'])
  for uid in movie['ratings']:
    movie['ratings'][uid] -= avg

100%|██████████| 87585/87585 [00:22<00:00, 3900.51it/s] 


In [ ]:
def cosine_similarity(a,b):
  keys = a.keys() & b.keys()
  n = 0
  for k in keys:
    n += a[k] * b[k]

  denom = (sum(v**2 for v in a.values()) * sum(v**2 for v in b.values())) ** 0.5

  if denom == 0:
    return 0

  return n / denom


In [ ]:
# 이 추천시스템은 어떤 영화가 입력으로 주어지면 이 영화의 별점 벡터와 pearson correlation coefficient가 높은 영화를 순서대로 추천합니다.
# Avengers: Endgame (2019)와 유사한 영화 출력
get_topk(122914,k=10,key='ratings',sim_func=cosine_similarity)

100%|██████████| 87585/87585 [07:13<00:00, 201.93it/s]


[(0.519061090106638, 'Avengers: Infinity War - Part I (2018)'),
 (0.3664577744776551, 'Thor: Ragnarok (2017)'),
 (0.35155550869331087, 'Spider-Man: Far from Home (2019)'),
 (0.30432195054029904, 'Captain America: Civil War (2016)'),
 (0.2956019903818523, 'Untitled Spider-Man Reboot (2017)'),
 (0.2791708194833043, 'Avengers, The (2012)'),
 (0.27791946188380273, 'Avengers: Age of Ultron (2015)'),
 (0.27513038770791415, 'Guardians of the Galaxy 2 (2017)'),
 (0.26963582846716316, 'Black Panther (2017)'),
 (0.26609152100739003, 'Doctor Strange (2016)')]